# Rekaman Prompt Hari 1 — Eksperimen EDA dengan AI

Pelatihan Machine Learning untuk Operasi PLTP · Hari 1, 15.00 · deck `materi-2b`

> **Rencana C — rekaman prompt.** Notebook ini dipakai bila AI coding agent tidak bisa
> diakses dari tempat pelatihan. Setiap langkah berisi *prompt* (sama dengan di slide) dan
> *kode yang dihasilkan AI* untuk prompt itu. Tugas Anda tetap sama dengan vibe coding:
> **baca kodenya, tebak hasilnya, jalankan, lalu periksa.**
>
> Begitu akses AI tersedia (VPN, hotspot, atau AI lokal perusahaan), ketik prompt yang sama
> ke agent dan bandingkan kodenya dengan rekaman ini. Tidak harus sama persis — yang harus
> sama adalah *angkanya*.

### Persiapan

> **Prompt:** Siapkan library dan baca `data/produksi_bulanan.csv` ke variabel `df`. Notebook dijalankan dari folder `notebook/`.

**Tebak dulu sebelum menjalankan sel di bawah:** berapa baris yang akan terbaca?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Notebook dijalankan dari folder notebook/, jadi data ada di ../data/
df = pd.read_csv("../data/produksi_bulanan.csv")
print("Data terbaca:", df.shape[0], "baris,", df.shape[1], "kolom")

### Tantangan 1 — gambaran umum

> **Prompt:** Tampilkan 5 baris pertama, ukuran tabel, tipe data, dan jumlah nilai kosong per kolom. Lalu hitung berapa unit yang berbeda di kolom `kode_unit`.

**Tebak dulu sebelum menjalankan sel di bawah:** 6 unit × 180 bulan = 1.080 baris. Apakah hasilnya 1.080? Berapa unit yang muncul?

In [ ]:
# 5 baris pertama
display(df.head())

# Tipe data dan jumlah nilai kosong per kolom
ringkas = pd.DataFrame({"tipe": df.dtypes.astype(str), "kosong": df.isna().sum()})
display(ringkas)

# Unit yang berbeda
print("Jumlah kode unit berbeda:", df["kode_unit"].nunique())
print(df["kode_unit"].value_counts())

**Periksa sebelum percaya:**

- [ ] 1.081 baris — satu lebih banyak dari seharusnya. Kenapa?
- [ ] 9 kode unit, bukan 6: `arl`, `btl`, `Btl` adalah salah ketik.
- [ ] `tanggal` bertipe `object` (teks), belum tanggal.
- [ ] 12 nilai kosong di `energi_aktual_mwh`.

### Tantangan 2 — sebaran CF per unit

> **Prompt:** Buat boxplot `cf_persen` per `kode_unit`. Urutkan unit dari median tertinggi. Tambahkan tabel rata-rata, minimum, dan maksimum CF per unit.

**Tebak dulu sebelum menjalankan sel di bawah:** berapa kotak yang akan muncul di grafik?

In [ ]:
# Urutkan unit berdasarkan median CF, dari tertinggi
urutan = df.groupby("kode_unit")["cf_persen"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 4))
df.boxplot(column="cf_persen", by="kode_unit", ax=ax)
ax.set_title("Sebaran CF per unit")
ax.set_xlabel("kode_unit")
ax.set_ylabel("CF (%)")
plt.suptitle("")
plt.show()

tabel = df.groupby("kode_unit")["cf_persen"].agg(["mean", "min", "max"]).round(1)
display(tabel.loc[urutan])

**Periksa sebelum percaya:**

- [ ] Grafik menampilkan 9 kotak — `arl`, `btl`, `Btl` berisi 1 titik saja. Kode jalan, tetapi menyesatkan.
- [ ] Perhatikan: AI menghitung `urutan` tetapi tidak memakainya di grafik. Membaca kode menangkap hal ini.
- [ ] CF minimum 2,6 % (CKR) — salah data, atau MO/FO panjang? Cek di `gangguan.csv`.

**Perbaikan sementara untuk tantangan berikutnya** (perbaikan permanen besok, Hari 2 10.00):

In [ ]:
df["unit"] = df["kode_unit"].str.upper()   # kolom baru; kolom asli tidak diubah
print("Unit setelah diseragamkan:", sorted(df["unit"].unique()))

### Tantangan 3 — tren dan musim

> **Prompt:** Hitung rata-rata `cf_persen` per bulan kalender (1–12) untuk semua unit, lalu buat grafik garis. Buat juga grafik total `energi_aktual_mwh` per tahun.

**Tebak dulu sebelum menjalankan sel di bawah:** bulan mana CF terendah? Apakah 2025 lengkap 12 bulan?

In [ ]:
cf_bulan = df.groupby("bulan")["cf_persen"].mean()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cf_bulan.plot(marker="o", ax=ax[0])
ax[0].set_title("Rata-rata CF per bulan kalender")
ax[0].set_xlabel("bulan")
ax[0].set_ylabel("CF (%)")
ax[0].set_xticks(range(1, 13))

energi_tahun = df.groupby("tahun")["energi_aktual_mwh"].sum() / 1000
energi_tahun.plot(kind="bar", ax=ax[1])
ax[1].set_title("Total energi aktual per tahun")
ax[1].set_ylabel("GWh")
plt.tight_layout()
plt.show()

print(cf_bulan.round(1))
print("Jumlah bulan per tahun (2025):", df[df["tahun"] == 2025]["bulan"].nunique())

**Periksa sebelum percaya:**

- [ ] `bulan` berupa angka, jadi urutannya 1–12 (bukan 1, 10, 11, 12, 2…).
- [ ] `sum()` diam-diam mengabaikan 12 nilai kosong — total energi tahun itu jadi lebih kecil. Tanyakan ke AI!
- [ ] Total per tahun memakai data yang masih berisi 1 baris duplikat.

### Tantangan 4 — pola gangguan

> **Prompt:** Dari `data/gangguan.csv`, hitung jumlah kejadian per `jenis`. Lalu khusus `jenis == "FO"`, buat grafik batang jumlah kejadian per `kode_unit` dan per `kategori_penyebab`.

**Tebak dulu sebelum menjalankan sel di bawah:** berapa total kejadian? Unit mana paling sering FO?

In [ ]:
gangguan = pd.read_csv("../data/gangguan.csv")
print("Total kejadian:", len(gangguan))
print(gangguan["jenis"].value_counts())

fo = gangguan[gangguan["jenis"] == "FO"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
fo["kode_unit"].value_counts().plot(kind="bar", ax=ax[0], title="Forced outage per unit")
fo["kategori_penyebab"].value_counts().plot(kind="bar", ax=ax[1], title="Forced outage per penyebab")
plt.tight_layout()
plt.show()

**Periksa sebelum percaya:**

- [ ] 200 kejadian: FO 99, FD 49, PD 35, MO 17.
- [ ] DRG paling sering FO (20) — tetapi unit punya kapasitas berbeda. Apakah 'paling sering' = 'paling berisiko'?
- [ ] Penyebab terbanyak: Sumur & Uap (semua jenis gangguan).

### Tantangan 5 — contoh hipotesis

> **Prompt:** Saya menduga unit yang lebih tua lebih sering forced outage. Gabungkan `gangguan.csv` dengan `unit_pembangkit.csv`, buat satu grafik dan satu angka ringkas. Sebutkan juga apa yang tidak bisa disimpulkan dari analisis ini.

**Tebak dulu sebelum menjalankan sel di bawah:** apakah hubungannya terlihat jelas?

In [ ]:
unit = pd.read_csv("../data/unit_pembangkit.csv")
fo_per_unit = fo["kode_unit"].value_counts().rename("jumlah_fo").reset_index()
gabung = unit.merge(fo_per_unit, on="kode_unit")

ax = gabung.plot(kind="scatter", x="tahun_operasi", y="jumlah_fo", s=80, figsize=(6, 4))
for _, r in gabung.iterrows():
    ax.annotate(r["kode_unit"], (r["tahun_operasi"], r["jumlah_fo"]), xytext=(5, 3), textcoords="offset points")
plt.title("Tahun mulai operasi vs jumlah forced outage")
plt.show()

print("Korelasi:", round(gabung["tahun_operasi"].corr(gabung["jumlah_fo"]), 2))
print("Keterbatasan: hanya 6 titik data; korelasi bukan sebab-akibat; kapasitas dan jumlah sumur tidak diperhitungkan.")

**Periksa sebelum percaya:**

- [ ] Hanya 6 unit — satu angka korelasi dari 6 titik sangat rapuh.
- [ ] Catat temuan ini di prompt log, termasuk keterbatasannya.